In [1]:
import pandas as pd
import yaml
from pathlib import Path

In [3]:
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config").is_dir())
with open(ROOT / "config" / "config.yaml", "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)
cfg = {k: str(ROOT / v) for k, v in cfg.items()}  # relative paths resolve against the project root

In [4]:
# Reading all df.
# 2024
P_A1_24_ready = pd.read_csv(cfg["P_A1_24_ready_csv_path"])
P_A2_24_ready = pd.read_csv(cfg["P_A2_24_ready_csv_path"])
P_B1_24_ready = pd.read_csv(cfg["P_B1_24_ready_csv_path"])

# 2025
P_A1_25_ready = pd.read_csv(cfg["P_A1_25_ready_csv_path"])
P_A2_25_ready = pd.read_csv(cfg["P_A2_25_ready_csv_path"])
P_B1_25_ready = pd.read_csv(cfg["P_B1_25_ready_csv_path"])

There are different names for the same products; we will standardise them after merging all plants.

In [5]:
# Create column comparison table
dfs = {
    'P_A1_24': P_A1_24_ready,
    'P_A2_24': P_A2_24_ready,
    'P_B1_24': P_B1_24_ready,
    'P_A1_25': P_A1_25_ready,
    'P_A2_25': P_A2_25_ready,
    'P_B1_25': P_B1_25_ready,
}

# Get all unique column names across all dataframes
all_columns = sorted(set().union(*[set(df.columns) for df in dfs.values()]))

# Create comparison dataframe
comparison_data = []
for col in all_columns:
    row = {'Column': col}
    for df_name in dfs.keys():
        row[df_name] = 'Yes' if col in dfs[df_name].columns else 'No'
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)

# Display in notebook
print(f"Total unique columns across all dataframes: {len(all_columns)}\n")
print(comparison_df.to_string(index=False))

# Export to CSV
output_file = Path("..") / "outputs" / "column_comparison.csv"
comparison_df.to_csv(output_file, index=False, encoding="utf-8")
print(f"\n✓ Exported to: {output_file}")


Total unique columns across all dataframes: 16

        Column P_A1_24 P_A2_24 P_B1_24 P_A1_25 P_A2_25 P_B1_25
    ad_auramix      No      No     Yes     Yes     Yes     Yes
ad_aut_dynamon      No     Yes      No      No     Yes      No
ad_complast_mr     Yes     Yes      No     Yes     Yes      No
 ad_dynamon_sx     Yes     Yes     Yes     Yes     Yes     Yes
       arid_04     Yes     Yes     Yes     Yes     Yes     Yes
    arid_12_20     Yes     Yes     Yes     Yes     Yes     Yes
     arid_8_14     Yes     Yes     Yes     Yes     Yes     Yes
  cement_3a_42     Yes      No     Yes      No      No     Yes
  cement_42_5r     Yes     Yes      No     Yes     Yes     Yes
    cement_4av     Yes     Yes      No     Yes      No      No
 corrective_02      No      No     Yes      No      No     Yes
          date     Yes     Yes     Yes     Yes     Yes     Yes
         plant     Yes     Yes     Yes     Yes     Yes     Yes
       product     Yes     Yes     Yes     Yes     Yes     Yes
       

In [6]:

# Diagnose: which source df has NaN in 'product' before merging?
source_dfs = {
    'P_A1_24': P_A1_24_ready,
    'P_A2_24': P_A2_24_ready,
    'P_B1_24': P_B1_24_ready,
    'P_A1_25': P_A1_25_ready,
    'P_A2_25': P_A2_25_ready,
    'P_B1_25': P_B1_25_ready,
}

for name, df in source_dfs.items():
    nan_count = df['product'].isna().sum()
    print(f"{name}: {nan_count} NaN in 'product'  (total rows: {len(df)})")


P_A1_24: 0 NaN in 'product'  (total rows: 6794)
P_A2_24: 0 NaN in 'product'  (total rows: 1408)
P_B1_24: 0 NaN in 'product'  (total rows: 1286)
P_A1_25: 0 NaN in 'product'  (total rows: 7524)
P_A2_25: 0 NaN in 'product'  (total rows: 1587)
P_B1_25: 0 NaN in 'product'  (total rows: 1246)


In [7]:
# Merge all dataframes into one
# Vertical concatenation (stack all rows), keeping all columns.

merged_df = pd.concat(
    [P_A1_24_ready, P_A2_24_ready, P_B1_24_ready,
     P_A1_25_ready, P_A2_25_ready, P_B1_25_ready],
    axis=0,  # 0 = stack rows vertically
    ignore_index=True,  # Reset index to sequential numbers
    sort=False  # Maintain column order as they appear
)

print(f"Merged dataframe shape: {merged_df.shape}")
print(f"Total rows: {merged_df.shape[0]}")
print(f"Total columns: {merged_df.shape[1]}")
print(f"\nColumns in merged dataframe:\n{merged_df.columns.tolist()}")
print(f"\nFirst few rows:\n{merged_df.head()}")
print(f"\nData types:\n{merged_df.dtypes}")
print(f"\nMissing values by column:\n{merged_df.isnull().sum()}")


Merged dataframe shape: (19845, 16)
Total rows: 19845
Total columns: 16

Columns in merged dataframe:
['date', 'plant', 'product', 'volume', 'cement_42_5r', 'cement_3a_42', 'cement_4av', 'arid_04', 'arid_8_14', 'arid_12_20', 'water', 'ad_complast_mr', 'ad_dynamon_sx', 'ad_aut_dynamon', 'ad_auramix', 'corrective_02']

First few rows:
         date plant           product  volume  cement_42_5r  cement_3a_42  \
0  2024-12-27  P_A1     HL-150/B/20/1     4.0         602.0           0.0   
1  2024-12-27  P_A1     HL-150/B/20/1     7.0        1049.0           0.0   
2  2024-12-27  P_A1    HM-20/B/20/I/1    10.0        2022.0           0.0   
3  2024-12-27  P_A1  HA-30/B/20/XC2/1     6.0        1751.0           0.0   
4  2024-12-27  P_A1     HL-150/B/20/1     5.0         749.0           0.0   

   cement_4av  arid_04  arid_8_14  arid_12_20   water  ad_complast_mr  \
0         0.0   4724.0     1590.0      1990.0   523.0             3.9   
1         0.0   8440.0     2794.0      3493.0   911.0   

In [8]:
# Move 'corrective_02' to the last column
merged_df = merged_df[[col for col in merged_df.columns if col != 'corrective_02'] + ['corrective_02']]

print(f"✓ Final columns:\n{merged_df.columns.tolist()}")


✓ Final columns:
['date', 'plant', 'product', 'volume', 'cement_42_5r', 'cement_3a_42', 'cement_4av', 'arid_04', 'arid_8_14', 'arid_12_20', 'water', 'ad_complast_mr', 'ad_dynamon_sx', 'ad_aut_dynamon', 'ad_auramix', 'corrective_02']


In [9]:
merged_df.isna().sum()

date                  0
plant                 0
product               0
volume                0
cement_42_5r       1286
cement_3a_42      10519
cement_4av         4119
arid_04               0
arid_8_14             0
arid_12_20            0
water                 0
ad_complast_mr     2532
ad_dynamon_sx         0
ad_aut_dynamon    16850
ad_auramix         8202
corrective_02     17313
dtype: int64

In [10]:
# Fill empty cells with 0.0
merged_df = merged_df.fillna(0.0)

In [11]:
merged_df.dtypes

date                  str
plant                 str
product               str
volume            float64
cement_42_5r      float64
cement_3a_42      float64
cement_4av        float64
arid_04           float64
arid_8_14         float64
arid_12_20        float64
water             float64
ad_complast_mr    float64
ad_dynamon_sx     float64
ad_aut_dynamon    float64
ad_auramix        float64
corrective_02     float64
dtype: object

In [12]:
# merged_df.to_csv(cfg["merged_ready_csv_path"], index=False, encoding="utf-8")
# Used just once. There is no need to save it again.

#### Let's proceed to standarize products names. Following the naming rules from the product dictionary. 

In [13]:
unique_product_count = merged_df["product"].nunique()
print(f"Number of unique products: {unique_product_count}")
print(f"\nUnique products:\n{merged_df['product'].unique()}")

Number of unique products: 173

Unique products:
<ArrowStringArray>
[               'HL-150/B/20/1',               'HM-20/B/20/I/1',
             'HA-30/B/20/XC2/1',             'HA-25/B/20/XC2/1',
       'IMPRESO.HA-25/B/14/IIa',             'HA-25/B/20/XC1/1',
             'HA-25/B/20/IIa/1',             'HA-25/F/14/IIa/1',
       'BOMBEABLE  HL-150/B/20',             'HA-30/B/20/IIa/1',
 ...
 'PILARES  MURO HA-25/B/20/XC2',   'IMPRESO METRO HM-20/F/14/I',
             'SECO  MD-250/S/4',       'IMPRESO HA-25/B/20/XC1',
      'IMPRESO  HA-30/B/20/IIa',   'M-5 MD-200/P/4 SIN ADITIVO',
           'HA-30/B/20/IV+Qb/1',                 'MD-250/P/4/1',
               'RES.H-200/B/20',             'HA-30/P/20/XA1/1']
Length: 173, dtype: str


In [14]:
# Export unique products to CSV for manual dictionary creation.
unique_products_df = pd.DataFrame({
    'product': merged_df['product'].unique()
})
output_path = Path("..") / "data" / "processed" / "UniqueProducts_Dictionary.csv"
unique_products_df.to_csv(output_path, index=False, encoding="utf-8")

print(f"✓ Exported {len(unique_products_df)} unique products to {output_path}")


✓ Exported 173 unique products to ..\data\processed\UniqueProducts_Dictionary.csv


In [15]:
# Looking at the unique names products. There are some products named with an xtra word/s mark it with a note. For example:
# CEMEX IMPRESO HA-25/B/14/XC2, IMPRESO.HA-30/F/14/IIa, PILARES  MURO HA-25/B/20/XC2, RES.H-200/B/20.
# However, to don't lose the notes. I will split using a "#" the note from the raw formula. PILARES MURO#HA-25-B-20-XC2.
# I will proceed to type manually the correct names. They are 173. I think it is worthy do it manually by a context owner (me, Jorge).

In [16]:
# Load the dictionary file
dictionary_path = Path("../docs/UniqueProducts_Dictionary_manual_ed.xlsx")
# Read the Excel file with the mapping
product_dict_df = pd.read_excel(dictionary_path, engine="openpyxl")

print("Dictionary loaded successfully!")
print(f"\nShape: {product_dict_df.shape}")
print(f"Columns: {product_dict_df.columns.tolist()}")
print(f"\nFirst few rows:\n{product_dict_df.head(10)}")


Dictionary loaded successfully!

Shape: (174, 2)
Columns: ['raw_name', 'update_name']

First few rows:
                                 raw_name                     update_name
0                                    A-15                         arid-15
1                                    A-20                         arid-20
2  ACCIONA ESTATICA PILOTE HA-30/L/14/XC2  ESTATICA PILOTE#HA-30-L-15-XC2
3                   AMAZON HA-25/B/20/IIa                      HA-25-B-20
4                      ARENA EN SECO A-04                         arid-04
5                       ARIDO   20   A-20                         arid-20
6                              ARIDO A-20                         arid-20
7                       ARLITA  D-400/B/4                ARLITA#D-400-B-4
8                  AURAMIX HA-25/B/20/IIa              AURAMIX#HA-25-B-20
9                  AUROMIX HA-25/B/20/XC2          AUROMIX#HA-25-B-20-XC2


### Create and apply the Mapping Dictionary
Convert the two columns into a dictionary where:
- **Key** = raw_name (the original product name)
- **Value** = updated_name (the standardized product name)


In [17]:
# Create a dictionary from raw_name -> updated_name
# This uses pandas.Series.to_dict() to convert two columns into a key-value mapping
product_mapping = dict(zip(product_dict_df['raw_name'], product_dict_df['update_name']))

print(f"✓ Mapping dictionary created!")
print(f"Total mappings: {len(product_mapping)}")
print(f"\nExample mappings (first 10):")
for i, (raw, updated) in enumerate(list(product_mapping.items())[:10]):
    print(f"  {i+1}. '{raw}' → '{updated}'")


✓ Mapping dictionary created!
Total mappings: 174

Example mappings (first 10):
  1. 'A-15' → 'arid-15'
  2. 'A-20' → 'arid-20'
  3. 'ACCIONA ESTATICA PILOTE HA-30/L/14/XC2' → 'ESTATICA PILOTE#HA-30-L-15-XC2'
  4. 'AMAZON HA-25/B/20/IIa' → 'HA-25-B-20'
  5. 'ARENA EN SECO A-04' → 'arid-04'
  6. 'ARIDO   20   A-20' → 'arid-20'
  7. 'ARIDO A-20' → 'arid-20'
  8. 'ARLITA  D-400/B/4' → 'ARLITA#D-400-B-4'
  9. 'AURAMIX HA-25/B/20/IIa' → 'AURAMIX#HA-25-B-20'
  10. 'AUROMIX HA-25/B/20/XC2' → 'AUROMIX#HA-25-B-20-XC2'


In [18]:
# Use the `map()` method to replace all product names in merged_df with their standardized versions from the dictionary.
# Apply the mapping to the 'product' column
# .map() replaces each value in merged_df['product'] with its corresponding value from the product_mapping dictionary.
merged_df['product'] = merged_df['product'].map(product_mapping)

print("✓ Mapping applied successfully!")
print(f"\nUpdated 'product' column:")
print(f"Total rows updated: {len(merged_df)}")
print(f"Unique standardized products: {merged_df['product'].nunique()}")
print(f"\nFirst 15 standardized product names:")
print(merged_df['product'].unique()[:15])


✓ Mapping applied successfully!

Updated 'product' column:
Total rows updated: 19845
Unique standardized products: 153

First 15 standardized product names:
<ArrowStringArray>
[          'HL-150-B-20',            'HM-20-B-20',        'HA-30-B-20-XC2',
        'HA-25-B-20-XC2',    'IMPRESO#HA-25-B-15',        'HA-25-B-20-XC1',
            'HA-25-B-20',        'HA-25-F-15-IIA', 'BOMBEABLE#HL-150-B-20',
            'HA-30-B-20',   'M -5 CON#MD-200-P-4',    'IMPRESO#HA-30-F-15',
       'SECO#D-250-S-20',        'RES#D-100-B-20',    'IMPRESO#HM-20-B-15']
Length: 15, dtype: str


### Verify the Results
Check for any missing mappings and inspect the updated data.


In [19]:
# Check for NaN values (these indicate unmapped products)
nan_count = merged_df['product'].isna().sum()

if nan_count > 0:
    print(f"⚠️  WARNING: {nan_count} rows have NaN values (unmapped products)")
    print("\nRows with unmapped products:")
    print(merged_df[merged_df['product'].isna()])
else:
    print("✅ SUCCESS: All products have been successfully mapped!")
    print(f"\nVerification Summary:")
    print(f"  • Total rows in merged_df: {len(merged_df)}")
    print(f"  • Unique standardized products: {merged_df['product'].nunique()}")
    print(f"  • Data type of 'product' column: {merged_df['product'].dtype}")
    print(f"\nSample of standardized products:")
    print(merged_df['product'].value_counts().head(10))


✅ SUCCESS: All products have been successfully mapped!

Verification Summary:
  • Total rows in merged_df: 19845
  • Unique standardized products: 153
  • Data type of 'product' column: str

Sample of standardized products:
product
HA-25-B-20                       3004
HA-25-B-20-XC2                   2520
HM-20-B-20                       2001
HA-25-B-20-XC1                   1338
CT-C12-F3                        1175
HA-30-B-20-XC2                    730
HA-25-F-15-IIA                    690
PILARES AMERIGO#HA-30-F-20-X0     531
HL-150-B-20                       522
PILOTES#HA-30-L-20-XC2            470
Name: count, dtype: int64


### Save the Updated Dataset.
Export the merged_df with standardized product names to a new Excel file.


In [20]:
# Save the updated dataframe with standardized product names
output_path = Path("..") / "data" / "processed" / "MergedData_standardized.csv"

merged_df.to_csv(output_path, index=False, encoding="utf-8")

print(f"✓ Updated dataset saved successfully!")
print(f"Location: {output_path}")
print(f"\nFile contains:")
print(f"  • Rows: {len(merged_df)}")
print(f"  • Columns: {len(merged_df.columns)}")
print(f"  • All product names standardized: ✅")


✓ Updated dataset saved successfully!
Location: ..\data\processed\MergedData_standardized.csv

File contains:
  • Rows: 19845
  • Columns: 16
  • All product names standardized: ✅


### Creating a new column "notes".


In [21]:
# Create the "note" column as string type, positioned after "product"
merged_df.insert(merged_df.columns.get_loc('product') + 1, 'note', pd.NA)

In [22]:
print(f"  • Columns: {len(merged_df.columns)}")
print( f" • Column names: {merged_df.columns.tolist()}")

  • Columns: 17
 • Column names: ['date', 'plant', 'product', 'note', 'volume', 'cement_42_5r', 'cement_3a_42', 'cement_4av', 'arid_04', 'arid_8_14', 'arid_12_20', 'water', 'ad_complast_mr', 'ad_dynamon_sx', 'ad_aut_dynamon', 'ad_auramix', 'corrective_02']


In [23]:
# For rows with "#": extract everything BEFORE "#" into "note", keep only AFTER it in "product".
mask = merged_df['product'].str.contains('#', na=False)

merged_df.loc[mask, 'note'] = merged_df.loc[mask, 'product'].str.split('#').str[0]
merged_df.loc[mask, 'product'] = merged_df.loc[mask, 'product'].str.split('#').str[1]

In [24]:
# Checking products name are correct.
from IPython.display import display
pd.set_option('display.max_rows', None)
display(pd.DataFrame(merged_df['product'].unique(), columns=['product']))
print (f"Data type of 'product' column: {merged_df['product'].dtype}")
# Checking note column values are correct too. 
from IPython.display import display
pd.set_option('display.max_rows', None)
display(pd.DataFrame(merged_df['note'].unique(), columns=['note']))
print (f"Data type of 'note' column: {merged_df['note'].dtype}")

,product
0,HL-150-B-20
1,HM-20-B-20
2,HA-30-B-20-XC2
3,HA-25-B-20-XC2
4,HA-25-B-15
5,HA-25-B-20-XC1
6,HA-25-B-20
7,HA-25-F-15-IIA
8,HA-30-B-20
9,MD-200-P-4


Data type of 'product' column: str


,note
0,NaN
1,IMPRESO
2,BOMBEABLE
3,M -5 CON
4,SECO
5,RES
6,RES BOMBEABLE
7,PILARES
8,M-5 CON
9,PILOTES MAURO


Data type of 'note' column: object


### Exporting the dataset

In [25]:
merged_df.to_csv(Path("..") / "data" / "export" / "consumption_dataset_ready.csv", index=False, encoding="utf-8")